# MyQuant - Data Exploration

在这个 Notebook 中，我们将探索如何加载和分析股票数据。

In [ ]:
# 导入所需库
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(''))))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.data.loader import DataLoader
from src.utils.visualization import PlotHelper

%matplotlib inline
plt.style.use('seaborn-v0_8')


In [ ]:
# 初始化数据加载器
loader = DataLoader('../config/data.yaml')
print(f"Configured universe: {loader.get_universe()}")


In [ ]:
# 测试数据加载（使用模拟数据）
def create_sample_data():
    dates = pd.date_range(start='2020-01-01', end='2023-12-31', freq='D')
    np.random.seed(42)
    base = 100
    returns = np.random.normal(0.001, 0.02, len(dates))
    prices = base * (1 + returns).cumprod()
    df = pd.DataFrame({
        'open': prices * (1 - np.random.normal(0.005, 0.01, len(dates))),
        'high': prices * (1 + np.random.normal(0.01, 0.005, len(dates))),
        'low': prices * (1 - np.random.normal(0.01, 0.005, len(dates))),
        'close': prices,
        'volume': np.random.randint(1000000, 10000000, len(dates))
    }, index=dates)
    return df

data = create_sample_data()
data.head()


In [ ]:
# 数据预览
print("Data shape:", data.shape)
print("\nSummary statistics:")
print(data.describe())


In [ ]:
# 绘制价格走势图
plt.figure(figsize=(12, 6))
plt.plot(data['close'], label='Close Price')
plt.plot(data['close'].rolling(20).mean(), label='MA20')
plt.plot(data['close'].rolling(60).mean(), label='MA60')
plt.title('Price Chart with Moving Averages')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# 计算收益率
returns = data['close'].pct_change()

plt.figure(figsize=(12, 6))
plt.hist(returns.dropna(), bins=50, density=True, alpha=0.7)
plt.title('Distribution of Daily Returns')
plt.xlabel('Daily Return')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()


In [ ]:
# 计算技术指标示例
def calculate_rsi(prices, period=14):
    delta = prices.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

rsi = calculate_rsi(data['close'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
ax1.plot(data['close'])
ax1.set_title('Price')
ax2.plot(rsi, color='orange')
ax2.axhline(y=70, color='red', linestyle='--')
ax2.axhline(y=30, color='green', linestyle='--')
ax2.set_title('RSI (14)')
ax2.grid(True)
plt.tight_layout()
plt.show()


## 下一步

- 运行 `scripts/download_data.py` 来下载真实的市场数据
- 探索不同的技术指标和特征
- 进入 `02_strategy_development.ipynb` 来学习如何开发和回测策略
